In [1]:
import torch
import pyro
import pyro.distributions as dist
import time

# Size of batch (increase this to see better parallelism)
N = 1_000_000

# Function to sample using pyro.plate
def sample_with_pyro(device):
    pyro.clear_param_store()
    start = time.time()
    with pyro.plate("data", N):
        x = pyro.sample("x", dist.Normal(torch.tensor(0., device=device), torch.tensor(1., device=device)))
    # For accurate GPU timing, sync
    if device == "cuda":
        torch.cuda.synchronize()
    return time.time() - start

# Function to sample using raw PyTorch
def sample_with_torch(device):
    start = time.time()
    loc = torch.tensor(0., device=device)
    scale = torch.tensor(1., device=device)
    samples = loc + scale * torch.randn(N, device=device)
    if device == "cuda":
        torch.cuda.synchronize()
    return time.time() - start

# Run CPU test
cpu_pyro_time = sample_with_pyro("cpu")
cpu_torch_time = sample_with_torch("cpu")

# Run GPU test
gpu_pyro_time = sample_with_pyro("cuda")
gpu_torch_time = sample_with_torch("cuda")

# Print results
print(f"CPU Pyro sampling time:  {cpu_pyro_time:.4f} sec")
print(f"CPU Torch sampling time: {cpu_torch_time:.4f} sec")
print(f"GPU Pyro sampling time:  {gpu_pyro_time:.4f} sec")
print(f"GPU Torch sampling time: {gpu_torch_time:.4f} sec")


/home/yaning/Documents/python_env/llm/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU Pyro sampling time:  0.0131 sec
CPU Torch sampling time: 0.0067 sec
GPU Pyro sampling time:  0.3216 sec
GPU Torch sampling time: 0.0002 sec


In [1]:
import torch
import pyro
import pyro.distributions as dist
import time
import torch.profiler

N = 1_000_000  # batch size

def sample_pyro(device):
    pyro.clear_param_store()
    with pyro.plate("data", N):
        x = pyro.sample("x", dist.Normal(torch.tensor(0., device=device), torch.tensor(1., device=device)))
    return x

def sample_torch(device):
    loc = torch.tensor(0., device=device)
    scale = torch.tensor(1., device=device)
    samples = loc + scale * torch.randn(N, device=device)
    return samples

# Profiler for Pyro sampling on GPU
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    sample_pyro("cuda")
    torch.cuda.synchronize()  # wait for GPU ops to finish

print("\n=== Pyro Sampling Profile ===")
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=100))

# # Profiler for raw Torch sampling on GPU
# with torch.profiler.profile(
#     activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
#     record_shapes=True,
#     profile_memory=True,
# ) as prof:
#     sample_torch("cuda")
#     torch.cuda.synchronize()

# print("\n=== Torch Sampling Profile ===")
# print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))


/home/yaning/Documents/python_env/llm/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== Pyro Sampling Profile ===
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::add         0.05%      75.552us         3.57%       5.961ms       5.961ms       9.056us        25.47%       9.056us       9.056us           0 b          

In [16]:
prof.key_averages().table()

'-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  \n                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  \n-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  \n                                            aten::empty         0.18%      46.850us         0.18%      46.850us       9.370us       0.000us         0.00%       0.000us       0.000us           8 b           8 b       3.81 Mb       3.